# LeetCode 3357 — Minimize the Maximum Adjacent Element Difference bằng Python

- Link: https://leetcode.com/problems/minimize-the-maximum-adjacent-element-difference/
- Độ khó: **Hard**
- Chủ đề: **Tìm kiếm nhị phân trên đáp án**, nhận xét **chọn `x = lo + d`, `y = hi − d`** (đẩy hai giá trị sát biên), kiểm tra từng **đoạn `-1`** bằng phân tích trường hợp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minDifference(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra đáp án đơn điệu theo `d` ⇒ **tìm kiếm nhị phân** "có thể đạt chênh lệch lớn nhất `≤ d` không?".
2. Hiểu vì sao với `d` cố định, chỉ cần thử **một** cặp: `x = lo + d`, `y = hi − d`, trong đó `lo`/`hi` là giá trị nhỏ/lớn nhất **trong các số đã biết đứng cạnh một `-1`**.
3. Chia mảng thành các **đoạn `-1` liên tiếp** và kiểm tra từng đoạn: đoạn dài 1 phải dùng **một** giá trị; đoạn dài `≥ 2` có thể dùng **cả `x` lẫn `y`** (cần `|x − y| ≤ d`).
4. Đừng quên các cặp **đã biết – đã biết** (chênh lệch cố định, là cận dưới).
5. Kiểm chứng bằng vét cạn mọi `(x, y)` + DP gán giá trị.

## 2. Đề bài & ví dụ

Mảng `nums`, một số phần tử bị thiếu (ký hiệu `-1`). Chọn **một lần** cặp số nguyên dương `(x, y)` và thay mỗi `-1` bằng `x` **hoặc** `y` (mỗi vị trí chọn riêng). Cực tiểu hóa **chênh lệch tuyệt đối lớn nhất giữa hai phần tử kề nhau**. Trả về giá trị nhỏ nhất đó.

**Ví dụ 1:** `[1,2,-1,10,8]` → `4` (chọn `(6, 7)` → `[1,2,6,10,8]`).
**Ví dụ 2:** `[-1,-1,-1]` → `0`.
**Ví dụ 3:** `[-1,10,-1,8]` → `1` (chọn `(11, 9)` → `[11,10,9,8]`).

**Ràng buộc:** `2 ≤ n ≤ 10^5`; `nums[i] = -1` hoặc `1 ≤ nums[i] ≤ 10^9`.

In [1]:
from typing import List

examples = [([1, 2, -1, 10, 8], 4), ([-1, -1, -1], 0), ([-1, 10, -1, 8], 1)]
for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp)

[1, 2, -1, 10, 8] -> kỳ vọng 4
[-1, -1, -1] -> kỳ vọng 0
[-1, 10, -1, 8] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Đánh giá một cặp `(x, y)` cố định — DP 2 trạng thái

Khi đã biết `(x, y)`, mỗi `-1` chỉ có 2 lựa chọn ⇒ DP "giá trị hiện tại là gì → chênh lệch lớn nhất nhỏ nhất tới đây". Đây là **oracle** để kiểm chứng.

In [2]:
def eval_xy(nums, x, y):
    """Chênh lệch kề lớn nhất nhỏ nhất khi mọi -1 được thay bằng x hoặc y."""
    prev = {v: 0 for v in ([nums[0]] if nums[0] != -1 else [x, y])}
    for a in nums[1:]:
        cand = [a] if a != -1 else [x, y]
        prev = {c: min(max(b, abs(c - p)) for p, b in prev.items()) for c in cand}
    return min(prev.values())

print(eval_xy([1, 2, -1, 10, 8], 6, 7), eval_xy([-1, 10, -1, 8], 11, 9))

4 1


### 3.2. Tìm kiếm nhị phân trên đáp án

Nếu đạt được chênh lệch lớn nhất `≤ d` thì cũng đạt `≤ d + 1` ⇒ tập `d` khả thi có dạng `[ans, +∞)`. Tìm `ans` bằng nhị phân, miễn có hàm `ok(d)` nhanh.

### 3.3. Các đại lượng chuẩn bị

- `base` = max `|nums[i] − nums[i+1]|` trên các cặp **cả hai đã biết** — không thể giảm, là cận dưới của đáp án.
- `lo`, `hi` = min / max của các số **đã biết đứng cạnh ít nhất một `-1`**. Nếu không có `-1` nào cạnh số đã biết (mảng toàn `-1` hoặc không có `-1`), đáp án là `base`.

In [3]:
def prepare(nums):
    base, lo, hi = 0, float('inf'), 0
    for a, b in zip(nums, nums[1:]):
        if a != -1 and b != -1:
            base = max(base, abs(a - b))
        elif a != -1 or b != -1:
            v = a if a != -1 else b
            lo, hi = min(lo, v), max(hi, v)
    return base, lo, hi

for nums, _ in examples:
    print(nums, '-> (base, lo, hi) =', prepare(nums))

[1, 2, -1, 10, 8] -> (base, lo, hi) = (2, 2, 10)
[-1, -1, -1] -> (base, lo, hi) = (0, inf, 0)
[-1, 10, -1, 8] -> (base, lo, hi) = (0, 8, 10)


### 3.4. Tách các đoạn `-1`

Mỗi đoạn `-1` liên tiếp được mô tả bằng `(trái, phải, độ dài)`, với `trái`/`phải` là số đã biết hai bên (hoặc `None` nếu đoạn chạm biên mảng).

In [4]:
def segments(nums):
    n, segs, i = len(nums), [], 0
    while i < n:
        if nums[i] != -1:
            i += 1
            continue
        j = i
        while j < n and nums[j] == -1:
            j += 1
        segs.append((nums[i - 1] if i > 0 else None, nums[j] if j < n else None, j - i))
        i = j
    return segs

print(segments([1, 2, -1, 10, 8]), segments([-1, 10, -1, 8]), segments([5, -1, -1, -1, 9, -1]))

[(2, 10, 1)] [(None, 10, 1), (10, 8, 1)] [(5, 9, 3), (9, None, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Giả sử ta muốn mọi chênh lệch `≤ d`, và `x ≤ y`.

- Có một `-1` đứng cạnh số `lo` ⇒ giá trị điền vào đó `≤ lo + d` ⇒ **giá trị nhỏ hơn `x` phải `≤ lo + d`**.
- Có một `-1` đứng cạnh số `hi` ⇒ giá trị điền vào `≥ hi − d` ⇒ **`y ≥ hi − d`**.

Đẩy `x` **lên** sát `lo + d` không làm hỏng gì: mọi số đã biết `a` mà `x` phải "với tới" đều thỏa `lo ≤ a ≤ x + d ≤ lo + 2d`, nên vẫn `|lo + d − a| ≤ d`; đồng thời `x` gần `y` hơn. Tương tự kéo `y` **xuống** `hi − d`. Vậy với `d` cố định chỉ cần thử **đúng một cặp** `(lo + d, hi − d)`!

Ví dụ 1: `lo = 2`, `hi = 10`. Với `d = 4`: `x = 6`, `y = 6` ⇒ `[1,2,6,10,8]` có chênh lệch lớn nhất `4` ✓. Với `d = 3`: `x = 5`, `y = 7` — ô giữa `2` và `10` cần một giá trị trong `[7, 5]` (rỗng) ✗.

## 5. Cách 1 — Brute Force: thử mọi `(x, y)` nhỏ + DP

Giá trị tối ưu không cần vượt quá `max(nums) + 1`, nên với mảng giá trị nhỏ ta thử mọi `1 ≤ x ≤ y ≤ max + 1` và chạy `eval_xy`.

In [5]:
def brute(nums):
    V = max(max(nums), 1) + 1
    return min(eval_xy(nums, x, y) for x in range(1, V + 1) for y in range(x, V + 1))

for nums, exp in examples:
    print(brute(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
0 | kỳ vọng 0
1 | kỳ vọng 1


**Độ phức tạp:** `O(V² · n)` — chỉ dùng làm oracle khi `V` nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `base`, `lo`, `hi`; nếu không có số đã biết nào cạnh `-1` ⇒ trả `base`.
2. Nhị phân `d ∈ [base, max(base, hi − lo)]` (với `d = hi − lo`, chọn `x = y = hi`… thực ra `x = lo + d = hi` luôn thỏa mọi đoạn).
3. `ok(d)`: đặt `x = lo + d`, `y = hi − d`; với mỗi đoạn `(a, b, L)`:
   - Hợp lệ nếu **một giá trị** `c ∈ {x, y}` cách `a` và `b` (nếu có) không quá `d`.
   - Hoặc nếu `L ≥ 2`, có cả hai bên, `|x − y| ≤ d`, và (`x` hợp `a`, `y` hợp `b`) hoặc ngược lại — đoạn dạng `x x … y y`.
   - Ngược lại `ok(d) = False`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Nhị phân `d` + kiểm tra bằng DP với cặp `(lo + d, hi − d)`

Dùng nhận xét ở mục 4 nhưng kiểm tra bằng `eval_xy` (DP 2 trạng thái) thay vì phân tích trường hợp — chậm hơn hằng số nhưng **khó sai**.

In [6]:
def bs_dp(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return base
    L, R = base, max(base, hi - lo)
    while L < R:
        d = (L + R) // 2
        if eval_xy(nums, lo + d, hi - d) <= d:
            R = d
        else:
            L = d + 1
    return L

for nums, exp in examples:
    print(bs_dp(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
0 | kỳ vọng 0
1 | kỳ vọng 1


### 7.2. Cách 3 — Nhị phân `d` + phân tích từng đoạn (lời giải chính)

In [7]:
def bs_case(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return base
    segs = segments(nums)

    def ok(d):
        x, y = lo + d, hi - d

        def fit(c, a, b):
            return (a is None or abs(c - a) <= d) and (b is None or abs(c - b) <= d)

        for a, b, L in segs:
            if fit(x, a, b) or fit(y, a, b):
                continue
            if L >= 2 and a is not None and b is not None and abs(x - y) <= d and \
               ((abs(x - a) <= d and abs(y - b) <= d) or (abs(y - a) <= d and abs(x - b) <= d)):
                continue
            return False
        return True

    L, R = base, max(base, hi - lo)
    while L < R:
        d = (L + R) // 2
        if ok(d):
            R = d
        else:
            L = d + 1
    return L

for nums, exp in examples:
    print(bs_case(nums), '| kỳ vọng', exp)

4 | kỳ vọng 4
0 | kỳ vọng 0
1 | kỳ vọng 1


### 7.3. Cách 4 — Không nhị phân trên toàn mảng: tính "yêu cầu" của từng đoạn

Mỗi đoạn tự nó cho một **ngưỡng tối thiểu** `need(seg)` — `d` nhỏ nhất để đoạn đó thỏa với cặp `(lo + d, hi − d)`; điều kiện của từng đoạn cũng đơn điệu theo `d`, nên đáp án là `max(base, max need(seg))`. Ở đây ta tính `need` bằng nhị phân **riêng cho từng đoạn** — cùng độ phức tạp, nhưng cho thấy bài toán **tách được** theo đoạn (hữu ích để debug: biết đoạn nào là "nút thắt").

In [8]:
def per_segment(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return base, []

    def seg_ok(seg, d):
        a, b, L = seg
        x, y = lo + d, hi - d
        fit = lambda c: (a is None or abs(c - a) <= d) and (b is None or abs(c - b) <= d)
        if fit(x) or fit(y):
            return True
        return L >= 2 and a is not None and b is not None and abs(x - y) <= d and \
            ((abs(x - a) <= d and abs(y - b) <= d) or (abs(y - a) <= d and abs(x - b) <= d))

    needs = []
    for seg in segments(nums):
        Lo, Hi = 0, hi - lo
        while Lo < Hi:
            d = (Lo + Hi) // 2
            if seg_ok(seg, d):
                Hi = d
            else:
                Lo = d + 1
        needs.append((seg, Lo))
    return max([base] + [v for _, v in needs]), needs

ans, needs = per_segment([1, 2, -1, 10, 8, -1, -1, 30])
print('đáp án:', ans)
for seg, v in needs:
    print('  đoạn', seg, '-> cần d >=', v)
print('khớp cách 3:', ans == bs_case([1, 2, -1, 10, 8, -1, -1, 30]))

đáp án: 10
  đoạn (2, 10, 1) -> cần d >= 4
  đoạn (8, 30, 2) -> cần d >= 10
khớp cách 3: True


Lưu ý: tính đơn điệu theo `d` của **từng đoạn** đúng vì khi `d` tăng, `x = lo + d` và `y = hi − d` dịch chuyển nhưng lý luận "đẩy sát biên" ở mục 4 vẫn áp dụng cho từng đoạn riêng — ta kiểm chứng điều này bằng test ngẫu nhiên ở mục 12.

## 8. TRACE từng bước

Trace cách 3 trên `[1, -1, -1, 10, 13]`: `base = 3`, `lo = 1`, `hi = 10`.

In [9]:
def trace(nums):
    base, lo, hi = prepare(nums)
    segs = segments(nums)
    print('base =', base, '| lo =', lo, '| hi =', hi, '| đoạn:', segs)
    L, R = base, max(base, hi - lo)
    while L < R:
        d = (L + R) // 2
        x, y = lo + d, hi - d
        verdict = eval_xy(nums, x, y) <= d
        print(f"  thử d = {d}: x = {x}, y = {y} -> {'được' if verdict else 'không'}")
        if verdict:
            R = d
        else:
            L = d + 1
    print('đáp án:', L, '| vét cạn:', brute(nums))

trace([1, -1, -1, 10, 13])

base = 3 | lo = 1 | hi = 10 | đoạn: [(1, 10, 2)]
  thử d = 6: x = 7, y = 4 -> được
  thử d = 4: x = 5, y = 6 -> được
  thử d = 3: x = 4, y = 7 -> được
đáp án: 3 | vét cạn: 3


Với `d = 3`: `x = 4`, `y = 7` ⇒ `[1, 4, 7, 10, 13]` — đoạn dài 2 dùng **cả hai** giá trị, điều mà một giá trị duy nhất không làm được.

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minDifference(self, nums: List[int]) -> int:
        n = len(nums)
        base, lo, hi = 0, 10 ** 18, 0
        for i in range(n - 1):
            a, b = nums[i], nums[i + 1]
            if a != -1 and b != -1:
                if abs(a - b) > base:
                    base = abs(a - b)
            elif a != -1 or b != -1:
                v = a if a != -1 else b
                if v < lo:
                    lo = v
                if v > hi:
                    hi = v
        if hi == 0:
            return base
        segs = []
        i = 0
        while i < n:
            if nums[i] != -1:
                i += 1
                continue
            j = i
            while j < n and nums[j] == -1:
                j += 1
            segs.append((nums[i - 1] if i > 0 else -1, nums[j] if j < n else -1, j - i))
            i = j

        def ok(d):
            x, y = lo + d, hi - d
            for a, b, L in segs:
                if a == -1:
                    a = b
                if b == -1:
                    b = a
                # một giá trị cho cả đoạn
                if (abs(x - a) <= d and abs(x - b) <= d) or (abs(y - a) <= d and abs(y - b) <= d):
                    continue
                # đoạn dài >= 2: x ... y hoặc y ... x
                if L >= 2 and abs(x - y) <= d and \
                   ((abs(x - a) <= d and abs(y - b) <= d) or (abs(y - a) <= d and abs(x - b) <= d)):
                    continue
                return False
            return True

        L, R = base, max(base, hi - lo)
        while L < R:
            d = (L + R) // 2
            if ok(d):
                R = d
            else:
                L = d + 1
        return L

for nums, exp in examples:
    got = Solution().minDifference(nums)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
0 OK
1 OK


**Mẹo cài đặt:** đoạn chạm biên chỉ có một hàng xóm; gán hàng xóm thiếu bằng hàng xóm còn lại (`a = b` hoặc `b = a`) giúp dùng chung công thức. Đoạn chạm biên không thể là toàn mảng vì khi đó `hi = 0` đã trả về sớm.

## 10. Vì sao thuật toán đúng

1. **Đơn điệu:** cấu hình đạt `≤ d` cũng đạt `≤ d + 1` ⇒ nhị phân hợp lệ.
2. **Chỉ cần cặp `(lo + d, hi − d)`:** giả sử có nghiệm `(x*, y*)`, `x* ≤ y*`, đạt `≤ d`. Số `lo` có một `-1` kề ⇒ một trong hai giá trị `≤ lo + d` ⇒ `x* ≤ lo + d`. Mọi số đã biết `a` kề một vị trí dùng `x*` thỏa `lo ≤ a ≤ x* + d ≤ lo + 2d` ⇒ `|lo + d − a| ≤ d`. Thay `x*` bằng `lo + d` giữ mọi ràng buộc với số đã biết; với `y*`: nếu `x* ≤ lo + d ≤ y*` thì `|y* − (lo + d)| ≤ |y* − x*|`. Lập luận đối xứng cho `y* → hi − d`. (Trường hợp `lo + d > hi − d` vẫn đúng: khi đó một giá trị duy nhất đã đủ — cả hai giá trị đều nằm trong `[hi − d, lo + d]`, khoảng cách tới mọi số đã biết cạnh `-1` `≤ d`.)
3. **Kiểm tra đoạn:** trong một đoạn `-1` giữa `a` và `b`, chỉ có các dạng: toàn một giá trị, hoặc chuyển đổi giữa `x` và `y` (cần `|x − y| ≤ d`). Đoạn dài 1 chỉ có một giá trị. Chuyển đổi nhiều lần không giúp gì hơn dạng `x…xy…y` vì chỉ hai đầu chạm số đã biết.
4. **Cận:** với `d = hi − lo`, `x = hi` cách mọi số trong `[lo, hi]` không quá `d` ⇒ mọi đoạn thỏa; kết hợp `base` cho cận phải `max(base, hi − lo)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi `(x, y)` + DP | `O(V² · n)` | `O(1)` |
| 2. Nhị phân + DP | `O(n log V)` | `O(1)` |
| 3. Nhị phân + phân tích đoạn | `O(n + s log V)` (`s` = số đoạn) | `O(s)` |
| 4. Nhị phân riêng từng đoạn | `O(n + s log V)` | `O(s)` |

`V = 10^9` ⇒ khoảng 30 vòng nhị phân.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'bs_dp': bs_dp, 'bs_case': bs_case, 'per_segment': lambda a: per_segment(a)[0],
         'submit': lambda a: Solution().minDifference(a)}
edge = [
    ([2, 1], 1),                 # không có -1
    ([-1, -1], 0),               # toàn -1
    ([5, -1], 0),                # đoạn chạm biên
    ([1, 10, -1], 9),            # base trội
    ([1, -1, -1, 4], 1),         # đoạn dài 2 dùng cả x và y
    ([3, -1, 5], 1),             # đoạn dài 1
    ([1, -1, -1, 10, 13], 3),
]
for nums, exp in examples + edge:
    assert brute(nums) == exp, (nums, brute(nums))
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums, f(nums))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
rng = random.Random(3357)
cnt = 0
for _ in range(1500):
    n = rng.randint(2, 7)
    nums = [rng.choice([-1, -1] + list(range(1, 13))) for _ in range(n)]
    exp = brute(nums)
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums, exp, f(nums))
    cnt += 1
print('Khớp vét cạn trên', cnt, 'mảng ngẫu nhiên.')

Khớp vét cạn trên 1500 mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [13]:
import time
N = 10 ** 5
cases = {
    'nửa -1, giá trị lớn': [rng.choice([-1, rng.randint(1, 10 ** 9)]) for _ in range(N)],
    'toàn -1 trừ hai đầu': [1] + [-1] * (N - 2) + [10 ** 9],
    'xen kẽ -1': [(-1 if i % 2 else rng.randint(1, 10 ** 9)) for i in range(N)],
}
for name, nums in cases.items():
    t = time.time(); a = Solution().minDifference(nums); t1 = time.time() - t
    t = time.time(); b = bs_dp(nums); t2 = time.time() - t
    assert a == b
    print(f"{name:22s}: {a:10d} | nộp {t1:.2f}s | bs_dp {t2:.2f}s")

nửa -1, giá trị lớn   :  991041973 | nộp 0.08s | bs_dp 1.88s


toàn -1 trừ hai đầu   :  333333333 | nộp 0.01s | bs_dp 2.90s


xen kẽ -1             :  498280090 | nộp 0.14s | bs_dp 2.13s


Ca "toàn `-1` trừ hai đầu": `lo = 1`, `hi = 10^9`, đoạn dài ⇒ dùng `x`, `y` với `|x − y| ≤ d`, `x − 1 ≤ d`, `10^9 − y ≤ d` ⇒ `d ≥ (10^9 − 1) / 3`.

In [14]:
print(Solution().minDifference([1] + [-1] * 5 + [10 ** 9]), -(-(10 ** 9 - 1) // 3))

333333333 333333333


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên các cặp đã biết đứng cạnh nhau (`base`)

In [15]:
def wrong_no_base(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return 0                       # SAI: bỏ qua base
    return bs_case(nums)

print('sai :', wrong_no_base([2, 1]), '| đúng:', Solution().minDifference([2, 1]))

sai : 0 | đúng: 1


**Vì sao sai:** mảng không có `-1` vẫn có chênh lệch `|2 − 1| = 1`. Ngoài ra nhị phân phải bắt đầu từ `base`, không phải 0.

### Lỗi 2 — Đoạn dài `≥ 2` dùng cả `x` và `y` nhưng quên điều kiện `|x − y| ≤ d`

In [16]:
def wrong_no_xy(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return base
    segs = segments(nums)
    def ok(d):
        x, y = lo + d, hi - d
        fit = lambda c, a, b: (a is None or abs(c - a) <= d) and (b is None or abs(c - b) <= d)
        for a, b, L in segs:
            if fit(x, a, b) or fit(y, a, b):
                continue
            if L >= 2 and a is not None and b is not None and \
               ((abs(x - a) <= d and abs(y - b) <= d) or (abs(y - a) <= d and abs(x - b) <= d)):
                continue                          # SAI: x và y đứng cạnh nhau!
            return False
        return True
    L, R = base, max(base, hi - lo)
    while L < R:
        d = (L + R) // 2
        L, R = (L, d) if ok(d) else (d + 1, R)
    return L

print('sai :', wrong_no_xy([3, -1, -1, 5]), '| đúng:', Solution().minDifference([3, -1, -1, 5]))

sai : 0 | đúng: 1


**Vì sao sai:** với `d = 0`: `x = 3`, `y = 5` ⇒ `[3, 3, 5, 5]` có chênh lệch `2` ở giữa — không phải `0`.

### Lỗi 3 — Coi mọi đoạn như đoạn dài 1 (chỉ cho phép một giá trị)

In [17]:
def wrong_single_per_segment(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return base
    segs = segments(nums)
    def ok(d):
        x, y = lo + d, hi - d
        fit = lambda c, a, b: (a is None or abs(c - a) <= d) and (b is None or abs(c - b) <= d)
        return all(fit(x, a, b) or fit(y, a, b) for a, b, L in segs)   # SAI: bỏ dạng x...y
    L, R = base, max(base, hi - lo)
    while L < R:
        d = (L + R) // 2
        L, R = (L, d) if ok(d) else (d + 1, R)
    return L

print('sai :', wrong_single_per_segment([1, -1, -1, 4]), '| đúng:', Solution().minDifference([1, -1, -1, 4]))

sai : 2 | đúng: 1


**Vì sao sai:** `[1, 2, 3, 4]` (dùng `x = 2`, `y = 3`) đạt `1`, trong khi một giá trị duy nhất cho cả đoạn cần `d = 2` (`[1, 3, 3, 4]`).

### Lỗi 4 — Lấy `lo`, `hi` trên **mọi** số đã biết thay vì chỉ số kề `-1`

In [18]:
def wrong_lo_hi_all(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return base
    known = [v for v in nums if v != -1]
    lo, hi = min(known), max(known)                # SAI
    segs = segments(nums)
    def ok(d):
        x, y = lo + d, hi - d
        fit = lambda c, a, b: (a is None or abs(c - a) <= d) and (b is None or abs(c - b) <= d)
        for a, b, L in segs:
            if fit(x, a, b) or fit(y, a, b):
                continue
            if L >= 2 and a is not None and b is not None and abs(x - y) <= d and \
               ((abs(x - a) <= d and abs(y - b) <= d) or (abs(y - a) <= d and abs(x - b) <= d)):
                continue
            return False
        return True
    L, R = base, max(base, hi - lo)
    while L < R:
        d = (L + R) // 2
        L, R = (L, d) if ok(d) else (d + 1, R)
    return L

bad = [1, -1, -1, 10, 13]
print('sai :', wrong_lo_hi_all(bad), '| đúng:', Solution().minDifference(bad))

sai : 4 | đúng: 3


**Vì sao sai:** số `13` không đứng cạnh `-1` nào, nhưng nếu tính vào `hi` thì `y = 13 − d` bị kéo quá cao: với `d = 3`, `y = 10` thay vì `7`, đoạn `1, _, _, 10` không còn xếp được `[1, 4, 7, 10]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cực tiểu hóa giá trị lớn nhất" ⇒ nghĩ ngay **nhị phân trên đáp án**.
- Có đúng hai "tham số tự do" `x`, `y` ⇒ tìm lý do để **cố định** chúng theo `d` (đẩy sát biên).
- Bài có ô trống `-1` ⇒ chia thành **đoạn**, mỗi đoạn chỉ quan tâm hai hàng xóm và độ dài.

## 15. Biến thể của bài

1. **Chỉ được chọn một giá trị `x`** cho mọi `-1` (dễ hơn): `ans = max(base, ⌈(hi − lo) / 2⌉)`.
2. **LC 2605 / bài "Minimize Max Difference" cùng họ** — nhị phân + kiểm tra tham lam.
3. Được chọn **`k` giá trị** thay vì 2 — khó hơn nhiều (bài toán phủ điểm).
4. Mảng vòng (phần tử cuối kề phần tử đầu).

In [19]:
# Biến thể 1: một giá trị duy nhất
def one_value(nums):
    base, lo, hi = prepare(nums)
    if hi == 0:
        return base
    return max(base, (hi - lo + 1) // 2)

def one_value_brute(nums):
    V = max(max(nums), 1) + 1
    return min(eval_xy(nums, x, x) for x in range(1, V + 1))

rng = random.Random(15)
for _ in range(800):
    nums = [rng.choice([-1, -1] + list(range(1, 13))) for _ in range(rng.randint(2, 7))]
    assert one_value(nums) == one_value_brute(nums), nums
print('Biến thể một giá trị: công thức max(base, ceil((hi-lo)/2)) khớp vét cạn.')

Biến thể một giá trị: công thức max(base, ceil((hi-lo)/2)) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi `(x, y)` + DP | `O(V² n)` | Oracle |
| 2 | Nhị phân + DP với cặp sát biên | `O(n log V)` | Khó sai |
| 3 | Nhị phân + phân tích đoạn | `O(n + s log V)` | **Nộp bài** |
| 4 | Ngưỡng riêng từng đoạn | `O(n + s log V)` | Tìm nút thắt |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[-1, 5, -1]`.

**Bài 2 (tính tay):** `[1, -1, 7]`.

**Bài 3 (điền TODO):** trả về luôn **cặp `(x, y)`** đạt đáp án.

**Bài 4 (tự viết lại):** biến thể **mảng vòng**.

**Bài 5 (mini challenge):** đếm số đoạn là "nút thắt" (đoạn có `need = ans`).

In [20]:
def pair_todo(nums):
    # TODO
    raise NotImplementedError

def circular_todo(nums):
    # TODO
    raise NotImplementedError

def bottlenecks_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Các `-1` chỉ kề số 5. (2) `x = y = 5`. (3) `0`.

**Bài 2.** (1) Đoạn dài 1 giữa 1 và 7. (2) Một giá trị phải cách cả hai `≤ d`. (3) `3` (giá trị 4).

**Bài 3.** (1) Tìm `d` như cách 3. (2) `x = lo + d`, `y = hi − d`. (3) Nếu `x > y` có thể chọn `x = y`; kiểm bằng `eval_xy`.

**Bài 4.** (1) Xoay mảng để bắt đầu tại một số đã biết. (2) Thêm phần tử đầu vào cuối. (3) Toàn `-1` ⇒ 0.

**Bài 5.** (1) Dùng `per_segment`. (2) Đếm `need == ans`. (3) `ans` có thể đến từ `base`, khi đó số nút thắt có thể là 0.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
assert Solution().minDifference([-1, 5, -1]) == 0 and Solution().minDifference([1, -1, 7]) == 3

def pair_of(nums):
    d = Solution().minDifference(nums)
    base, lo, hi = prepare(nums)
    if hi == 0:
        return d, (1, 1)
    x, y = lo + d, hi - d
    if x > y:
        x = y = (x + y) // 2 if eval_xy(nums, (x + y) // 2, (x + y) // 2) <= d else x
    return d, (x, y)

rng = random.Random(19)
for _ in range(500):
    nums = [rng.choice([-1, -1] + list(range(1, 13))) for _ in range(rng.randint(2, 7))]
    d, (x, y) = pair_of(nums)
    assert x >= 1 and y >= 1 and eval_xy(nums, x, y) == d, (nums, d, x, y)
print('Bài 1: 0 | Bài 2: 3 | Bài 3: cặp trả về luôn đạt đáp án.')

Bài 1: 0 | Bài 2: 3 | Bài 3: cặp trả về luôn đạt đáp án.


In [22]:
# Bài 4, 5
def circular(nums):
    if all(v == -1 for v in nums):
        return 0
    s = next(i for i, v in enumerate(nums) if v != -1)
    rot = nums[s:] + nums[:s]
    return Solution().minDifference(rot + [rot[0]])

def circular_brute(nums):
    V = max(max(nums), 1) + 1
    best = float('inf')
    for x in range(1, V + 1):
        for y in range(x, V + 1):
            idx = [i for i, v in enumerate(nums) if v == -1]
            for mask in range(1 << len(idx)):
                arr = nums[:]
                for t, i in enumerate(idx):
                    arr[i] = y if mask >> t & 1 else x
                best = min(best, max(abs(arr[i] - arr[(i + 1) % len(arr)]) for i in range(len(arr))))
    return best

def bottlenecks(nums):
    ans, needs = per_segment(nums)
    return sum(1 for _, v in needs if v == ans)

rng = random.Random(4)
for _ in range(300):
    nums = [rng.choice([-1, -1] + list(range(1, 8))) for _ in range(rng.randint(2, 6))]
    assert circular(nums) == circular_brute(nums), nums
print('Bài 4 khớp vét cạn | Bài 5 ví dụ:', bottlenecks([1, 2, -1, 10, 8, -1, -1, 30]))

Bài 4 khớp vét cạn | Bài 5 ví dụ: 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
base = max |a-b| trên cặp đã biết; lo/hi = min/max số đã biết kề -1
nếu không có: return base
nhị phân d trong [base, max(base, hi-lo)]:
    x, y = lo + d, hi - d
    mỗi đoạn (a, b, L): một giá trị hợp cả a,b  hoặc  (L>=2, |x-y|<=d, x~a & y~b hoặc ngược lại)
```

**Checklist:**

- [ ] Đã tính `base` từ các cặp đã biết–đã biết?
- [ ] `lo`, `hi` chỉ lấy từ số **kề `-1`**?
- [ ] Đoạn dài `≥ 2` cho phép `x…y` với điều kiện `|x − y| ≤ d`?
- [ ] Đoạn chạm biên (chỉ một hàng xóm) xử lý đúng?
- [ ] Mảng toàn `-1` hoặc không có `-1` trả về `base`?